In [1]:
import numpy as np
import random
import sys
import sklearn
import time
from rdkit import Chem
import matplotlib.pyplot as plt
from IPython.core.display import HTML
from IPython.display import SVG
import torch.nn as nn
import pandas as pd
from rdkit.Chem import AllChem
from rdkit import RDLogger
import os
import rdkit
from torchvision.transforms import ToTensor, ToPILImage
import re
import torchvision.transforms as transforms
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
import torch

import torch.nn.functional as F
from torch.nn import Sequential, Linear, ReLU, GRU
import torch.nn
import os.path as osp
sys.path.append("~/Su")
import SupContrast

from SupContrast.util import TwoCropTransform, AverageMeter
from SupContrast.util import adjust_learning_rate, warmup_learning_rate
from SupContrast.util import set_optimizer, save_model
from SupContrast.networks.resnet_big import SupConResNet
from SupContrast.losses import SupConLoss
from sklearn.cluster import KMeans

try:
    import rdkit
    from rdkit.Chem import PandasTools, AllChem as Chem, Descriptors,rdmolfiles
    from rdkit import Chem
    from rdkit import rdBase
    from rdkit.Chem.rdchem import HybridizationType
    from rdkit import RDConfig
    from rdkit.Chem import ChemicalFeatures
    from rdkit.Chem.rdchem import BondType as BT
    rdBase.DisableLog('rdApp.error')
except ImportError:
    rdkit = None
    print("No Rdkit")


from torch.nn import BatchNorm1d
from torch.utils.data import Dataset
import torch.optim as optim

from sklearn.ensemble import RandomForestClassifier
random.seed(4)
torch.manual_seed(4)
np.random.seed(4)

In [ ]:
#import cv2
from torchvision import datasets, transforms
print(torch.__version__, torch.version.cuda)

2.5.1 11.8


### Load dataframes for image datasets and data frames with labels

In [ ]:
all_df_fmp=pd.read_parquet("FMP_CP/fmp_full.parq")
all_df_fmp=all_df_fmp[["Molecule name","WELL","ImgNumber"]]
all_df_fmp.rename(columns={"Molecule name":"Metadata_EOS","WELL":"Metadata_Well","ImgNumber":"ImgID"},inplace=True)
all_df_fmp["Site"]="FMP"
all_df_fmp["ImgID"]=all_df_fmp["ImgID"].astype(str)
all_df_fmp.head(1)# site ID

In [ ]:
all_df=pd.read_csv("euos/euopen_upgraded_layout.csv",delimiter=";")
#4CimgUSC96768_P24.png
name_key="Metadata_EOS"
all_df=all_df.rename(columns={"Unnamed: 0":"ImgID"})

all_df.head(1)

In [ ]:
combined_df = pd.concat([all_df, all_df_fmp], ignore_index=True)
all_df=combined_df

In [ ]:
print(len(all_df))
all_df = all_df[
    ~((all_df["Site"] == "USC") &
      (all_df["Metadata_Plate"] == "B1007") &
      (all_df["Metadata_Batch"] == "R2") &
      (all_df["Metadata_Well"] == "E23"))
]
print("Well with many artifiact images filtered out.")
len(all_df)

In [ ]:
#all_df=all_df[all_df.Site=="FMP"]
all_df["Bioactive"]=(all_df[name_key]!="DMSO").astype(int)
len(all_df)

In [ ]:
bmoa_ohe_matrix=pd.read_csv("FMP_CP/bmoa_ohe_matrix_fmp.csv",delimiter=";",index_col=0)
btarget_ohe_matrix=pd.read_csv("FMP_CP/btarget_ohe_matrix_fmp.csv",delimiter=";",index_col=0)
#mesh_ohe_matrix=pd.read_csv("FMP_CP/mesh_ohe_matrix_fmp.csv",delimiter=";",index_col=0)# from year ~2022. fewer annotations in pubchem found
mesh_ohe_matrix=pd.read_csv("FMP_CP/mesh_ohe_matrix_fmp_UPD.csv",delimiter=";",index_col=0)# year ~2025
pnd_ohe_matrix=pd.read_csv("FMP_CP/pnd_ohe_matrix_fmp.csv",delimiter=";",index_col=0)
print(len(mesh_ohe_matrix),len(pnd_ohe_matrix),len(bmoa_ohe_matrix),len(btarget_ohe_matrix))
bmoa_ohe_matrix.head(1)

In [ ]:
train_df_mesh=all_df.copy()
train_df_mesh=train_df_mesh[train_df_mesh[name_key].isin(mesh_ohe_matrix.index)]
len(train_df_mesh)

In [ ]:
train_df_mesh.Metadata_EOS.value_counts()

In [ ]:
train_df_pnd=all_df.copy()
train_df_pnd=train_df_pnd[train_df_pnd[name_key].isin(pnd_ohe_matrix.index)]
len(train_df_pnd)

In [ ]:
train_df_bmoa=all_df.copy()
train_df_bmoa=train_df_bmoa[train_df_bmoa[name_key].isin(bmoa_ohe_matrix.index)]
len(train_df_bmoa)

In [ ]:
train_df_btarget=all_df.copy()
train_df_btarget=train_df_btarget[train_df_btarget[name_key].isin(btarget_ohe_matrix.index)]
len(train_df_btarget)

In [ ]:
max_classes=int(mesh_ohe_matrix.sum(axis=1).max())
bmoa_max=int(bmoa_ohe_matrix.sum(axis=1).max())
btarget_max=int(btarget_ohe_matrix.sum(axis=1).max())
pnd_max=int(pnd_ohe_matrix.sum(axis=1).max())
print(mesh_ohe_matrix.sum(axis=1).mean(),bmoa_ohe_matrix.sum(axis=1).mean(),btarget_ohe_matrix.sum(axis=1).mean(),pnd_ohe_matrix.sum(axis=1).mean())
max_classes,bmoa_max,btarget_max,pnd_max

In [ ]:
mesh_ohe_matrix.head(1)

### Setup torch datasets and loaders

In [ ]:
dose_id="concentration_mM"
image_id="ImgID"
resize_images=False
write_images=False
write_path='...'
read_path='euopen224_upd/'
name_id=name_key
all_df["NameCategorical"] = all_df[name_id].astype('category')
all_df["NameCategorical_codes"] = all_df["NameCategorical"].cat.codes
namecat_pos=list(all_df.columns).index("NameCategorical_codes")
name_pos=list(all_df.columns).index(name_id)
mock_pos=list(all_df.columns).index("Bioactive")
imgid_pos=list(all_df.columns).index(image_id)

inputX_key="Anno"
well_key="Metadata_Well"
well_pos=list(all_df.columns).index(well_key)
res1,res2=224,224
img_format=".png"
multi_label=True
multi_label_pnd=multi_label
same_site_neigh=False
class merged_img_ds(Dataset):
    """Face Landmarks dataset."""

    def __init__(self, df=all_df,transform=None,img_transform=None,alt_view=None,multi=False,multi_pnd=False,multi_bmoa=False,multi_btarget=False,single_labels=None):
        """
        Args:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied
                on a sample.
        """
        self.input = df
        #self.root_dir = root_dir
        self.transform = transform
        self.img_transform = img_transform
        self.multi=multi
        self.multi_pnd=multi_pnd
        self.multi_bmoa=multi_bmoa
        self.multi_btarget=multi_btarget
        self.all_the_single_labels=single_labels

    def __len__(self):
        return len(self.input)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

       
        rowX=self.input.iloc[idx]
        name = rowX.iloc[name_pos] 
        image_index=rowX.iloc[imgid_pos]
        site=rowX.Site
        well=rowX[well_key]
        ##4CimgUSC96768_P24.png"
        if site=="FMP":
            img_path="fmp/4Cimg224_"+str(image_index)+'.png'
            img_path="fmp/4Cimg"+str(image_index)+"_"+well+'.png'#4Cimg18538_F20.png FMP has different naming scheme
        else:
            img_path=read_path+site+"/4Cimg"+str(image_index)+"_"+well+'.png'
        image=Image.open(img_path).convert('RGB')  
        neigh=self.input[self.input[name_id]==name]#replicate image
        if same_site_neigh:
            neigh=neigh[neigh.Site==site]
        neigh=list(neigh.index)
        name_ilocs=[]
        for locs in neigh:#name in locs
            name_ilocs.append(self.input.index.get_loc(locs))#gets ilocs
        name_ilocs.remove(idx) 
        neigh_idx=random.choice(name_ilocs)
        neigh_idx_image=str(self.input.iloc[neigh_idx,imgid_pos] )
        neigh_well=str(self.input.iloc[neigh_idx,well_pos] )
        if same_site_neigh:
            neigh_site=site
        else:
            neigh_site=str(self.input.iloc[neigh_idx].Site)
        if neigh_site=="FMP":
            neigh_path="fmp/4Cimg224_"+str(neigh_idx_image)+'.png'
            neigh_path="fmp224_upd/fmp/4Cimg"+str(neigh_idx_image)+"_"+neigh_well+'.png'#4Cimg18538_F20.png
        else:
            neigh_path=read_path+neigh_site+"/4Cimg"+str(neigh_idx_image)+"_"+neigh_well+'.png'
        neigh_img=Image.open(neigh_path).convert('RGB')   
        if resize_images:
            image=image.resize((res1, res2),resample=Image.NEAREST)
        if write_images:
            cv2.imwrite(write_path+str(idx)+'.png',image)     
        conc=np.array(float(0)) # old dummy
        if self.multi==False and self.multi_pnd==False and self.multi_bmoa==False and self.multi_btarget==False:
            if type(self.all_the_single_labels)==pd.core.frame.DataFrame:#
                v1=  self.all_the_single_labels.loc[name].most_frequent_label_index
                inputX_values=np.array(v1)
            else:
                inputX_values=np.array(0)
        if self.multi:#
            inputX_values=  mesh_ohe_matrix.loc[name]
            inputX_values=inputX_values[inputX_values.values != 0].index
            
            inputX_values=[mesh_ohe_matrix.columns.get_loc(col) for col in inputX_values]
            container=np.zeros((max_classes))
            #print(container,hit, inputX_values)
            container[:len(inputX_values)]=inputX_values
            inputX_values=container
        if self.multi_pnd:#
            inputX_values= pnd_ohe_matrix.loc[name]
            inputX_values=inputX_values[inputX_values.values != 0].index
            
            inputX_values=[pnd_ohe_matrix.columns.get_loc(col) for col in inputX_values]
            container=np.zeros((pnd_max))
            #print(container,hit, inputX_values)
            container[:len(inputX_values)]=inputX_values
            inputX_values=container
        if self.multi_bmoa:#
            inputX_values= bmoa_ohe_matrix.loc[name]
            inputX_values=inputX_values[inputX_values.values != 0].index
            
            inputX_values=[bmoa_ohe_matrix.columns.get_loc(col) for col in inputX_values]
            container=np.zeros((bmoa_max))
            #print(container,hit, inputX_values)
            container[:len(inputX_values)]=inputX_values
            inputX_values=container
        if self.multi_btarget:#
            inputX_values= btarget_ohe_matrix.loc[name]
            inputX_values=inputX_values[inputX_values.values != 0].index
            
            inputX_values=[btarget_ohe_matrix.columns.get_loc(col) for col in inputX_values]
            container=np.zeros((btarget_max))
            #print(container,hit, inputX_values)
            container[:len(inputX_values)]=inputX_values
            inputX_values=container
            #inputX_key="Targets"
                
        assay_role= rowX.iloc[mock_pos]
        assay_role=np.array(assay_role)   
        target_class = np.array(0)#dummy
        
        if self.img_transform=="transform":
            aug = self.img_transform(image)
            sample = {'image': image, 'Bioactive': assay_role,"path":img_path, inputX_key:inputX_values, "Name":name,"Aug":aug}            
            sample = self.transform(sample,image_transform=True)
        if self.img_transform==None:
            sample = {'image': image, 'Bioactive': assay_role,"path":img_path, inputX_key:inputX_values, "Name":name}
           
        if self.img_transform=="neighbor":#replicates as positve
           sample = {'image': image, "path":img_path,inputX_key:inputX_values, "Name":name,"Aug":neigh_img,"anno2":target_class}
           sample = self.transform(sample,image_transform="neighbor")
        if self.img_transform=="neighbor_code":#weakly supervised
            name_code = np.array(rowX.iloc[namecat_pos])
            sample = {'image': image,"path":img_path,inputX_key:inputX_values, "Name":name,"Aug":neigh_img,"anno2":target_class,"name_code":name_code,"imgID":image_index}
            sample = self.transform(sample,image_transform="neighbor_code")
        

        return sample
class ToTensor(object):
    """Convert ndarrays in sample to Tensors."""

    def __call__(self, sample,image_transform=False):
        if image_transform:
            image, path,inputX_values, name,aug,anno2 = sample['image'], sample['path'],sample[inputX_key], sample['Name'],sample["Aug"],sample["anno2"]
        if image_transform=="neighbor_code":
            image, path,inputX_values, name,aug,anno2,name_cat,imgID = sample['image'], sample['path'],sample[inputX_key], sample['Name'],sample["Aug"],sample["anno2"],sample["name_code"],sample["imgID"]
        image=np.asarray(image)
        image = image.transpose(2, 0, 1)
        image=torch.from_numpy(image.copy())        
        if image_transform=="neighbor" or "neighbor_code":
            aug=np.asarray(aug)
            aug = aug.transpose(2, 0, 1)
            aug=torch.from_numpy(aug.copy())
        if image_transform=="neighbor":
            return {'image': image,'path':0,"dummy":0, inputX_key:torch.from_numpy(inputX_values),"Name":name,"Aug":aug}
        if image_transform=="neighbor_code":
            return {'image': image,'path':path,"dummy":0, inputX_key:torch.from_numpy(inputX_values),"Name":name,"Aug":aug,"NameCat":name_cat,"imgID":imgID}
train_dataset=merged_img_ds(df=all_df,transform=ToTensor(),img_transform="neighbor_code")#weakly supervised
train_dataset_nonans_bmoa=merged_img_ds(df=train_df_bmoa,transform=ToTensor(),img_transform="neighbor",multi_bmoa=multi_label)#single_labels=df_most_frequent_bmoa
train_dataset_nonans_btarget=merged_img_ds(df=train_df_btarget,transform=ToTensor(),img_transform="neighbor",multi_btarget=multi_label_pnd)#,single_labels=df_most_frequent_targets

train_dataset_nonans_mesh=merged_img_ds(df=train_df_mesh,transform=ToTensor(),img_transform="neighbor",multi=multi_label)
train_dataset_nonans_pnd=merged_img_ds(df=train_df_pnd,transform=ToTensor(),img_transform="neighbor",multi_pnd=multi_label_pnd)
train_dataset_nonans_bmoa[0],train_dataset_nonans_btarget[0];

In [ ]:
train_dataset_nonans_bmoa[0]["Anno"]

In [ ]:
plt.imshow(train_dataset_nonans_mesh[0]["image"].numpy().transpose())

In [ ]:
plt.imshow(train_dataset_nonans_pnd[5]["image"].numpy().transpose())

In [ ]:
bs=8*65
bsz2=bs
bs_unsup=bs
bsz=bs
bsz_unsup=bs_unsup
workers=1
print(bsz,bsz_unsup)
train_loader = torch.utils.data.DataLoader(train_dataset, shuffle=True,pin_memory=True,batch_size=bs_unsup,drop_last=True, num_workers=workers)
train_loader_feat = torch.utils.data.DataLoader(train_dataset, shuffle=False,pin_memory=True, num_workers=workers)
train_loader_nonans_bmoa = torch.utils.data.DataLoader(train_dataset_nonans_bmoa, shuffle=True,pin_memory=True,batch_size=bs
                                                  ,drop_last=True, num_workers=workers)
train_loader_nonans_btarget = torch.utils.data.DataLoader(train_dataset_nonans_btarget, shuffle=True, pin_memory=True,
                                                  batch_size=bs,
                                                  drop_last=True, num_workers=workers)
train_loader_nonans_mesh = torch.utils.data.DataLoader(train_dataset_nonans_mesh, shuffle=True,pin_memory=True,batch_size=bs
                                                  ,drop_last=True, num_workers=workers)
train_loader_nonans_pnd = torch.utils.data.DataLoader(train_dataset_nonans_pnd, shuffle=True,pin_memory=True,batch_size=bs
                                                  ,drop_last=True, num_workers=workers)
len(train_loader)

#### Loss function

In [ ]:
device0=0
device_str="cuda:{0}".format(device0)#0
ct=0

print(device_str)
device = torch.device(device_str if torch.cuda.is_available() else 'cpu')

class SupConLoss_own(nn.Module):#single-label supervised contrastive learning 
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.07, contrast_mode='all',#increase temp to 0.1???
                 base_temperature=0.07,device=device):
        super(SupConLoss_own, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf
        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

class SupConLoss_multi(nn.Module):#my multi-label version of SupCon
    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.07,device=device):
        super(SupConLoss_multi, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            #labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            labels_sub=labels[(labels!=0).sum(1)>1]
            polys=((labels!=0).sum(1)>1).nonzero()
            labels0 = labels[:,0].contiguous().view(-1, 1)
            mask=torch.eq(labels0, labels0.T).float()
            others=list(range(len(labels)))
            for n,i in enumerate(labels_sub):
                poly_pos=polys[n].item()
                othersX=others.copy()
                othersX.remove(poly_pos)
                multi2=i[i!=0]
                for nxt_class in multi2:
                    for otherpos, other_row in enumerate(labels[othersX]):
                        other_row=other_row[other_row!=0]
                        if (other_row==nxt_class).any():
                            mask[poly_pos,othersX[otherpos]]=1
                            if other_row.shape[0]==1:
                                mask[othersX[otherpos],poly_pos ]=1
            mask = mask.float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask.sum(1)

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

### Pick and setup encoder

In [ ]:
from torchvision import models
eb1=models.efficientnet_b1(weights="EfficientNet_B1_Weights.IMAGENET1K_V2")
del eb1.classifier
for param in eb1.parameters():
    param.requires_grad = False
pcount=0
for n,param in enumerate(eb1.parameters()):
    pcount+=1
    if n<5:
        print(param.requires_grad )
print(param.requires_grad )
pcount

In [ ]:
from transformers import AutoImageProcessor, ResNetForImageClassification
resnet50_1_5 = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
del resnet50_1_5.classifier #this classifier actually also flatttens your data. We only want the image encoder
#resnet50_1_5.to(device)
from transformers import AutoImageProcessor, ResNetForImageClassification
resnet152 = ResNetForImageClassification.from_pretrained("microsoft/resnet-152")
del resnet152.classifier #this classifier actually also flatttens your data
#resnet50_1_5.to(device)

In [ ]:
resnet50_1_5

In [ ]:
from transformers import AutoImageProcessor, ResNetForImageClassification
resnet152 = ResNetForImageClassification.from_pretrained("microsoft/resnet-152")
del resnet152.classifier #this classifier actually also flatttens your data
#resnet50_1_5.to(device)

In [ ]:
print("Different image encoders may require different ways of forwarding outputs")
class SupConCVmodel(nn.Module):#example
    #backbone + projection head
    def __init__(self, name='ResNet50_1.5', head='mlp',dim_in2=2048, feat_dim=224):
        super(SupConCVmodel, self).__init__()
        model_fun, dim_in = model_dict[name]
        if name.endswith("_1.5"):
            self.encoder = model_fun
            if head == 'linear':
                self.classifier = nn.Linear(dim_in, feat_dim)
            elif head == 'mlp':
                self.encoder.classifier = nn.Sequential(nn.Flatten(),
                    nn.Linear(dim_in, dim_in2),
                    nn.ReLU(inplace=True),
                    nn.Linear(dim_in2, feat_dim)
                )


    def forward(self, x):
        encoding=self.encoder(x)['logits']
        feat = F.normalize(encoding, dim=1)
        #feat =self.encoder(x)#feat = F.normalize(self.encoder(x), dim=1)
        #feat = F.normalize(self.head(feat), dim=1)#classifier
        return feat

model_dict = {'ResNet50_1.5': [resnet50_1_5, 2048],'ResNet152_1.5': [resnet152, 2048],'eb1': [eb1, 1280]}#


In [ ]:
import torch.backends.cudnn as cudnn

def set_model(ddp_gpu=True,gpu=True,multi_label=multi_label):
        model = SupConCVmodel(name='ResNet50_1.5', feat_dim=224)
        criterion = SupConLoss_own(temperature=0.07)#.5 in Con CP paper (Perakis et al., 2021)
        if multi_label:
            criterion=SupConLoss_multi(temperature=0.07)
        if ddp_gpu:
            model.encoder = torch.nn.DataParallel(model.encoder,device_ids=list(range(device0,device0+8)))
        if gpu:#single-gpu
            model = model.cuda(device=device)
            criterion = criterion.cuda(device=device)
        cudnn.benchmark = True

        return model, criterion
print("Multi label is set to: ", multi_label)
model, criterion = set_model(ddp_gpu=True,gpu=True)
molc_criterion=SupConLoss_own(temperature=0.07)
molc_criterion = molc_criterion.cuda(device=device)
bsz=bs#112#num_mesh_classes
bs2=bs
bsz2=bs2
print(bsz,bsz2)

In [ ]:
pretrain=False
###PATHx="/content/drive/My Drive/cp/geometric/models/model1_full_bs128_ac50_64conv1.pt"

just_test=False
exp_pathx=False
save,save_all_ep=True,True
PATHx_save="models/model_supbs"+str(bs)+"selfsup_bs"+str(bs_unsup)
PATHx_save+="EUOS_4CJ_multisupcon_BMT_R50_1-5_weights.pt"
idisk=""
PATHy=""#pretrained model path
print(PATHx_save)
print(PATHy)

In [ ]:
outfile="embeddings/emb"+PATHx_save.split("/")[-1][5:-3]+".csv"
logfile="log"+PATHx_save.split("/")[-1][5:-3]
outfile,logfile

In [ ]:
if pretrain:
  model.load_state_dict(torch.load(PATHy,map_location="cuda:0"))#map_location="cpu" torch.device("cpu") 
  print("Pretrain is ON",PATHy)
else:
    print("Pretrain is OFF")
optimizer = torch.optim.Adam(model.parameters(), lr=10**-2.5)#10**-2.5

In [ ]:
model.to(device);

### Training and testing functions

In [ ]:
with open(outfile, "w+") as g:
    g.write("")
def trainNtest2(loader,outfile=outfile,):#train model and write embeddings the RF can use for testing
    count=0
    model.train()
    seperator=";"
    y_available=True
    loss_all=0
    with open(outfile, "w+") as g:
        g.write("")
    for data in loader:
      count+=1
      y_output=[]
      ylabel=[]
      entropy=[]
      latentnames=[]
      img_pos=[]
      error = 0
      data = [im for im in data.values()]
      images=data[0].to(device).float()
      aug=data[5].to(device).float()
      labels=data[3].cuda(device=device, non_blocking=True) #.cuda(device=device,non_blocking=True)#.int()
      images = torch.cat([images, aug], dim=0)
      optimizer.zero_grad()
      features = model(images)
      f1, f2 = torch.split(features, [bsz, bsz], dim=0)
      features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
      loss = criterion(features, labels)
      loss.backward()
      loss_all += loss.item()  # * data.num_graphs
      optimizer.step()
      if y_available:
              y_output.append(features.cpu().detach().numpy())
              ylabel.append(labels.cpu().detach().numpy())
              entropy.append(loss.cpu().detach().numpy())
              img_pos.append(data[2])
      with open(outfile, "a+") as g:
          for no,point in enumerate(data[4]):
              youtput_str=[str(value) for value in y_output[0][no][0]]
              g.writelines(str(point)+seperator+str(0)+seperator+str(entropy[0]))#str(y_output[n][no][0])[1:-1]+seperator
              g.write(";")
              g.write(";".join(youtput_str))
              g.write("\n")
      if count==2:
        if debug:
            break
    return loss_all/(len(loader)*bs)



In [ ]:
with open(outfile, "w+") as g:
    g.write("")
def trainNtest(loader,outfile=outfile,):
    count=0
    model.train()
    seperator=";"
    y_available=True
    loss_all=0
    with open(outfile, "w+") as g:
        g.write("")
    for data in loader:
      count+=1
      y_output=[]
      ylabel=[]
      entropy=[]
      latentnames=[]
      img_pos=[]
      error = 0
      data = [im for im in data.values()]
      images=data[0].to(device).float()
      aug=data[5].to(device).float()
      labels=data[3].cuda(device=device, non_blocking=True) #.cuda(device=device,non_blocking=True)#.int()
      images = torch.cat([images, aug], dim=0)
      optimizer.zero_grad()
      features = model(images)
      f1, f2 = torch.split(features, [bsz, bsz], dim=0)
      features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
      loss = criterion(features, labels)
      loss.backward()
      loss_all += loss.item()  # * data.num_graphs
      optimizer.step()
      if y_available:
              y_output.append(features.cpu().detach().numpy())
              ylabel.append(labels.cpu().detach().numpy())
              entropy.append(loss.cpu().detach().numpy())
              img_pos.append(data[2])
      with open(outfile, "a+") as g:
          for no,point in enumerate(data[4]):
              youtput_str=[str(value) for value in y_output[0][no][0]]
              g.writelines(str(point)+seperator+str(ylabel[0][no])+seperator+str(entropy[0]))#str(y_output[n][no][0])[1:-1]+seperator
              g.write(";")
              g.write(";".join(youtput_str))
              g.write("\n")
      if count==2:
        if debug:
            break
    return loss_all/(len(loader)*bs)

def dsRF(outfile=outfile):
    dft=pd.read_csv(outfile,header=None,index_col=0,delimiter=";")
    dft["Molecules"]=dft.index
    groups = [df for _, df in dft.groupby('Molecules')]
    random.shuffle(groups)
    test_dfX_2=pd.concat(groups).reset_index(drop=True)
    unique=len(set(test_dfX_2.Molecules))
    tr_part2=round(len(test_dfX_2.Molecules)*0.8)#*36
    skX2,skY2=test_dfX_2.iloc[:tr_part2,2:-1],test_dfX_2[1][:tr_part2]
    skX_test2,skY_test2=test_dfX_2.iloc[tr_part2:,2:-1],test_dfX_2[1][tr_part2:]
    rf = RandomForestClassifier(n_estimators=1000, random_state=0,n_jobs=20)
    rf.fit(skX2, skY2)
    score=rf.score(skX_test2,skY_test2)#99% train
    print(score, "Accuracy")
    
    return score

In [ ]:
def testwrite(loader,outfile=outfile):#calculate embeddings of images for downstream models. 
    print(outfile)
    count=0
    model.eval()
    seperator=";"
    y_available=True
    loss_all=0
    with open(outfile, "w+") as g:
        g.write("")
    for data in loader:
      count+=1
      y_output=[]
      ylabel=[]
      entropy=[]
      latentnames=[]
      img_pos=[]
      error = 0
      data = [im for im in data.values()]
      images=data[0].to(device).float()
      aug=data[5].to(device).float()
      labels=data[3].to(device)
      images = torch.cat([images, aug], dim=0)
      with torch.no_grad():
          features = model(images)
          f1, f2 = torch.split(features, [bs, bs], dim=0)
          features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
          loss = criterion(features)  
      if y_available:
              y_output.append(features.cpu().detach().numpy())
              ylabel.append(labels.cpu().detach().numpy())
              entropy.append(loss.cpu().detach().numpy())
              img_pos.append(data[2])
      with open(outfile, "a+") as g:
          for no,point in enumerate(data[4]):
              youtput_str=[str(value) for value in y_output[0][no][0]]
              g.writelines(point+seperator+str(ylabel[0][no])+seperator+str(entropy[0]))
              g.write(";")
              g.write(";".join(youtput_str))
              g.write("\n")
    return loss_all/(len(loader)*bs)
#testwrite(train_loader_feat)

In [ ]:
start_feat,end_feat=2,-1
moi=bmoa_ohe_matrix.copy()#matrix of interest; pick a label set used for evaluating the RF
def dsmultiRF(outfile=outfile,moi=moi):#RF accuracy used for early-stopping  
    dft=pd.read_csv(outfile,header=None,index_col=0,delimiter=";")
    dft["Molecules"]=dft.index
    dft=dft[dft.Molecules.isin(moi.index)]
    moi=moi[moi.index.isin(dft["Molecules"])]
    skf = GroupKFold(n_splits=5)
    groups = [df for _, df in dft.groupby('Molecules')]
    random.shuffle(groups)
    dX=pd.concat(groups).reset_index(drop=True)
    #unique=len(set(dX.Molecules))
    #tr_part2=round(len(dX.Molecules)*0.8)
    for i, (train_index, test_index) in enumerate(skf.split(dX, groups=dX["Molecules"])):
        trainX=dX.iloc[train_index].copy()
        skX2=trainX.iloc[:,start_feat:end_feat]
        skY2=moi.loc[trainX.Molecules]

        testX=dX.iloc[test_index].copy()
        skX_test2=testX.iloc[:,start_feat:end_feat]
        skY_test2=moi.loc[testX.Molecules]
        forest = RandomForestClassifier(random_state=1,n_jobs=20)
        multi_target_forest = MultiOutputClassifier(forest, n_jobs=1)
        multi_target_forest.fit(skX2, skY2)#.predict(skX)
        score=multi_target_forest.score(skX_test2,skY_test2)#99% train
        #pred=multi_target_forest.predict(skX_test2)
        print(score, "Accuracy")
        break
    return score
#dsmultiRF()

In [ ]:
moi.head(1)

### Start training

In [ ]:
def train(loader):
    model.train()
    loss_all = 0
    for data in loader:
        data = [im for im in data.values()]
        images=data[0].cuda(device=device,non_blocking=True).float()
        aug=data[5].cuda(device=device,non_blocking=True).float()
        labels=data[3].cuda(device=device,non_blocking=True)#.int()
        #print(images.shape,aug.shape)
        images = torch.cat([images, aug], dim=0)
        optimizer.zero_grad()
        features = model(images)
        #print(features.shape,images.shape)
        f1, f2 = torch.split(features, [bsz, bsz], dim=0)
        features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
        loss = criterion(features,labels)
        loss.backward()
        loss_all += loss.item() 
        optimizer.step()
        if debug:
          break
    return loss_all / (len(loader)*bsz)#
def selfsup_train(loader=train_loader):
    model.train()
    loss_all = 0
    for data in loader:
        data = [im for im in data.values()]
        images=data[0].cuda(device=device,non_blocking=True).float()
        aug=data[5].cuda(device=device,non_blocking=True).float()
        #labels=data[3].cuda(device=device,non_blocking=True)#.int()
        molc_labels=data[6].cuda(device=device,non_blocking=True)#.int()
        images = torch.cat([images, aug], dim=0)
        optimizer.zero_grad()
        features = model(images)
        #print(features.shape,images.shape)
        f1, f2 = torch.split(features, [bsz2, bsz2], dim=0)
        features = torch.cat([f1.unsqueeze(1), f2.unsqueeze(1)], dim=1)
        loss = molc_criterion(features,molc_labels)
        loss.backward()
        loss_all += loss.item()
        optimizer.step()
        if debug:
          break
    return loss_all / (len(loader)*bsz2)#

best_score = None
ep=500
debug=False
if debug:
  ep=2
if just_test:
  ep=1
for epoch in range(1,ep):
    loss=0
    start=time.time()
    lr=0      
    scoreRF=0
    if not just_test:
          #loss += unsup_train(epoch)
          if epoch % 2 == 0:
                pnd_loss =train(loader=train_loader_nonans_btarget)
                #torch.cuda.empty_cache() #sometimes helps
                #torch.cuda.synchronize()
                mesh_loss = trainNtest(train_loader_nonans_bmoa)
                loss+=pnd_loss
                loss+=mesh_loss
                sup_loss=str(loss)
                print("Supervised loss", loss)
                print("Calculate Accuracy:")
                scoreRF=dsmultiRF()
          else:
                pnd_loss =train(loader=train_loader_nonans_btarget)
                #torch.cuda.empty_cache()
                #torch.cuda.synchronize()
                mesh_loss = train(train_loader_nonans_bmoa)
                loss+=pnd_loss
                loss+=mesh_loss
                sup_loss=str(loss)
                print("Supervised loss", loss)
          #torch.cuda.empty_cache()
          #torch.cuda.synchronize()
          loss += selfsup_train()
    if save:
      torch.save(model.state_dict(), PATHx_save)
      #print("You are saved")
      if save_all_ep:
        PATHx_saveN=PATHx_save[:-3]+str(epoch)+PATHx_save[-3:]
        torch.save(model.state_dict(), PATHx_saveN)
      if best_score is None or scoreRF > best_score:
        best_score = scoreRF
        if save:
            torch.save(model.state_dict(), PATHx_save[:-3]+"_best"+PATHx_save[-3:])
    
    end=time.time()
    diff_time=abs(start-end)
    print(diff_time)
    if just_test:
      loss=0
    loss_tr.append(loss)
    print('Epoch: {:03d}, Loss: {:.7f}'.format(epoch,  loss))
    with open(logfile,"a+") as g:
            g.writelines("BMoA loss "+ str(mesh_loss)+"\n")
            g.writelines("BTarget loss "+ str(pnd_loss)+"\n")
            g.writelines("Supervised loss "+ str(sup_loss)+"\n")
            g.writelines(str(diff_time)+"\n")
            g.writelines('Epoch: {:03d}, Loss: {:.7f}'.format(epoch, loss)+"\n")
            g.writelines(str(scoreRF)+"accuracy \n")

    
